In [ ]:
import os
os.chdir('../')

In [ ]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from utils.fid import FIDInception
from tqdm import tqdm

# ------------ config ------------
DEVICE = torch.device('cuda:0')
VAL_NPZ = 'prompts/mscoco2014_val.npz'
IMG_ROOT = Path('/dataset/val2014')
OUT_DIR  = Path('mscoco2014_fid')
OUT_PATH = OUT_DIR / 'coco2014_val_fid_stats.pt'
LIMIT = 10000   # 전체 쓰려면 None 또는 주석 처리
# --------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
net = FIDInception(device=DEVICE).eval()

valid = np.load(VAL_NPZ, allow_pickle=True)['arr_0']
if LIMIT is not None:
    valid = valid[:LIMIT]

feats = []
with torch.inference_mode():
    for img_id, _ in tqdm(valid, desc='Extracting Inception features'):
        p = IMG_ROOT / f'COCO_val2014_{int(img_id):012d}.jpg'
        with Image.open(p) as im:
            im = im.convert('RGB')
        feats.append(net([im]).detach().cpu())  # (1, 2048)

feats = torch.cat(feats, dim=0).to(torch.float32)   # (N, 2048)
mu    = feats.mean(dim=0)                           # (2048,)
sigma = torch.cov(feats.T, correction=1)            # (2048, 2048) — unbiased

# ✅ Torch .pt로 통계만 저장
torch.save({'mu': mu.cpu(), 'sigma': sigma.cpu(), 'n': int(feats.shape[0])}, OUT_PATH)
print('done:', mu.shape, sigma.shape, '->', OUT_PATH)


Extracting Inception features:  68%|██████▊   | 6788/10000 [01:33<00:45, 70.24it/s]